# Lab 4 — Evaluating a RAG Agent

Building a grounded agent is half the job — **proving** it behaves correctly is the other half. This lab evaluates the RAG **prompt agent** from **Lab 2** against a small, hand-written test set and scores three behaviours that matter for any internal knowledge assistant:

| Metric | Question it answers |
|---|---|
| **Groundedness** | Did the answer come from the retrieved documents (not invented)? |
| **Citation** | Did the answer cite a source? |
| **Out-of-scope refusal** | Did the agent refuse questions outside the knowledge base? |

You'll run two styles of evaluation:

1. **Deterministic checks** (no LLM) — canary-token grounding, citation presence, refusal detection.
2. **LLM-as-judge** — an LLM scores correctness/groundedness for nuanced cases.

> **Prerequisites:** the Lab 2 prompt agent (`labs-prompt-agent`) grounded with its `AzureAISearchTool` and the seeded `contoso-outdoors` index.


## 1. Define the evaluation set

Each test case has a question, the **canary token** we expect a grounded answer to contain (or `None` for out-of-scope cases), and whether the agent should **refuse**. Swap these for questions over your own knowledge base.

In [2]:
TEST_CASES = [
    # In-scope: a grounded answer must contain the canary token and a citation.
    {"question": "What is your return policy? Include any item codes.",
     "expect_canary": "TR-CANARY-7821", "should_refuse": False},
    {"question": "How long does standard shipping take and is there a promo code?",
     "expect_canary": "SHIP-CANARY-4493", "should_refuse": False},
    {"question": "How do I clean my tent and what SKU is the re-waterproofing kit?",
     "expect_canary": "TENT-CANARY-9067", "should_refuse": False},
    # Out-of-scope: the agent should refuse (no canary expected).
    {"question": "What is the capital of France?",
     "expect_canary": None, "should_refuse": True},
    {"question": "Write me a poem about the stock market.",
     "expect_canary": None, "should_refuse": True},
]
print(f"{len(TEST_CASES)} test cases ready.")

5 test cases ready.


## 2. Collect agent responses

We call the Lab 2 prompt agent (`labs-prompt-agent`) through the OpenAI-compatible Responses API for each question and capture the text output.

In [3]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

load_dotenv()

AGENT_NAME = "labs-prompt-agent"
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
credential = DefaultAzureCredential()

project = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project.get_openai_client()


def ask_agent(question: str) -> str:
    response = openai_client.responses.create(
        extra_body={"agent_reference": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=question,
    )
    return (response.output_text or "").strip()


for case in TEST_CASES:
    case["answer"] = ask_agent(case["question"])
    print(f"Q: {case['question']}\nA: {case['answer'][:160]}...\n")

Q: What is your return policy? Include any item codes.
A: Customers may return any item within 30 days of delivery. Items should be unused and include the original packaging. Every accepted return also ships back with ...

Q: How long does standard shipping take and is there a promo code?
A: Standard shipping arrives in 3–5 business days, and it’s free on orders over $50. There’s also a promo code for a one-time free overnight upgrade: SHIP-CANARY-4...

Q: How do I clean my tent and what SKU is the re-waterproofing kit?
A: Clean tent fabric with lukewarm water and non-detergent soap. The re-waterproofing kit is stocked under SKU TENT-CANARY-9067.【5:0-source】...

Q: What is the capital of France?
A: I don’t have that information in the Contoso Outdoors knowledge base....

Q: Write me a poem about the stock market.
A: I don’t have that information in the Contoso Outdoors knowledge base....



## 3. Deterministic evaluation (no LLM)

Fast, cheap, and reproducible checks:

- **Grounded** — for in-scope cases, the expected canary token appears in the answer.
- **Cited** — the answer references a source (a URL or the word "source").
- **Refused** — for out-of-scope cases, the answer signals refusal and contains **no** canary token.

In [ ]:
import re

REFUSAL_MARKERS = (
    "don't have", "do not have", "can't help", "cannot help", "out of scope",
    "outside", "not able to", "no information", "unable to", "not covered",
)


def looks_refused(text: str) -> bool:
    low = text.lower()
    return any(m in low for m in REFUSAL_MARKERS)


def is_cited(text: str) -> bool:
    return bool(re.search(r"https?://|source", text, re.IGNORECASE))


rows = []
for case in TEST_CASES:
    ans = case["answer"]
    if case["should_refuse"]:
        passed = looks_refused(ans) and "CANARY" not in ans
        detail = "refused" if looks_refused(ans) else "did NOT refuse"
    else:
        grounded = case["expect_canary"] in ans
        cited = is_cited(ans)
        passed = grounded and cited
        detail = f"grounded={grounded}, cited={cited}"
    rows.append((case["question"], "PASS" if passed else "FAIL", detail, ans))

for question, res, detail, ans in rows:
    print(f"[{res}] {question}")
    print(f"       why : {detail}")
    print(f"       answer: {ans[:200].replace(chr(10), ' ')}")
    print()

passed = sum(1 for _, r, _, _ in rows if r == "PASS")
print(f"Deterministic pass rate: {passed}/{len(rows)} ({passed / len(rows):.0%})")


[PASS] What is your return policy? Include any item codes.
       why : grounded=True, cited=True
       answer: Customers may return any item within 30 days of delivery. Items should be unused and include the original packaging. Every accepted return also ships back with a complimentary Contoso TrailRunner stic

[PASS] How long does standard shipping take and is there a promo code?
       why : grounded=True, cited=True
       answer: Standard shipping arrives in 3–5 business days, and it’s free on orders over $50. There’s also a promo code for a one-time free overnight upgrade: SHIP-CANARY-4493【5:0-source】

[PASS] How do I clean my tent and what SKU is the re-waterproofing kit?
       why : grounded=True, cited=True
       answer: Clean tent fabric with lukewarm water and non-detergent soap. The re-waterproofing kit is stocked under SKU TENT-CANARY-9067.【5:0-source】

[PASS] What is the capital of France?
       why : refused
       answer: I don’t have that information in the Contoso

## 4. LLM-as-judge (groundedness & correctness)

Deterministic checks catch the obvious cases; an LLM judge handles nuance (paraphrasing, partial answers). We ask a model to score each in-scope answer on a 1–5 scale for how well it is **grounded in and consistent with** the retrieved source content.

This is the same idea as a *correctness judge* in a RAG benchmark — useful when comparing two implementations (e.g. custom RAG vs a low-code tool).

In [6]:
from azure.ai.projects import AIProjectClient

model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4.1")
project = AIProjectClient(endpoint=project_endpoint, credential=credential)
judge = project.get_openai_client()

JUDGE_PROMPT = (
    "You are an evaluation judge for a knowledge-base assistant. "
    "Given a QUESTION and the assistant's ANSWER, rate from 1 to 5 how well the answer is "
    "grounded, specific, and helpful. Reply with ONLY the integer score."
)

scores = []
for case in TEST_CASES:
    if case["should_refuse"]:
        continue  # judge only in-scope answers
    result = judge.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"QUESTION: {case['question']}\nANSWER: {case['answer']}"},
        ],
    )
    raw = result.choices[0].message.content.strip()
    match = re.search(r"[1-5]", raw)
    score = int(match.group()) if match else 0
    scores.append(score)
    print(f"score={score}  Q: {case['question'][:50]}")

if scores:
    print(f"\nAverage LLM judge score: {sum(scores) / len(scores):.2f} / 5")

score=4  Q: What is your return policy? Include any item codes
score=4  Q: How long does standard shipping take and is there 
score=5  Q: How do I clean my tent and what SKU is the re-wate

Average LLM judge score: 4.33 / 5


## 5. Summary

You now have a repeatable, two-layer evaluation harness:

- **Deterministic** grounding / citation / refusal checks — cheap regression gates you can run in CI.
- **LLM-as-judge** correctness — for nuanced scoring and comparing implementations.

**Extend it for your own knowledge base:**

- Replace `TEST_CASES` with real questions and expected facts from your documents.
- Add metrics such as **embedding similarity** between answer and source chunk, or **latency** per query.
- Track scores over time (each model or prompt change) to catch regressions before release.
- Use the same harness to benchmark a custom RAG agent against a low-code alternative.